# VisionTrack: Exploratory Data Analysis & Preprocessing

This notebook covers the complete exploratory data analysis (EDA) and data validation required by VisionTrack:
1. **Data Loading & Video Stream Exploration**: Frame analysis, resolution, FPS, and codec inspection.
2. **Exploratory Data Analysis (EDA)**: Person detection distribution, confidence thresholds, and bounding box geometry.
3. **YOLO Preprocessing Pipeline**: Resizing, letterboxing, normalization, and tensor transformation.
4. **Annotation Verification**: Validation of YOLO format label files (class x_center y_center width height).

In [1]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'utils' else Path.cwd()))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils.data_loader import VideoStreamLoader, ImageDatasetLoader
from utils.preprocessing import letterbox, normalize_frame, preprocess_for_yolo, StaticCameraMotionGate
from models.yolo_person_detection import PersonDetector

print('Environment initialized successfully!')

## 1. Video Stream Exploration
We inspect the primary video streams to verify frame rates, resolutions, and total durations.

In [2]:
stream_path = 'mov-1.mov' if os.path.exists('mov-1.mov') else '../mov-1.mov'
loader = VideoStreamLoader(stream_path)
meta = loader.get_metadata()
print('Stream Metadata:')
for k, v in meta.items():
    print(f'  {k}: {v}')

## 2. Preprocessing Pipeline Verification
YOLO requires inputs with shape [1, 3, 640, 640] normalized between [0.0, 1.0].

In [3]:
# Read a test frame
for frame_idx, pts, frame in loader:
    if frame_idx == 30:
        sample_frame = frame
        break
loader.release()

tensor, ratio, (pad_w, pad_h) = preprocess_for_yolo(sample_frame, target_size=(640, 640))
print(f'Original Frame Shape: {sample_frame.shape}')
print(f'YOLO Input Tensor Shape: {tensor.shape}')
print(f'Pixel Range: [{tensor.min():.2f}, {tensor.max():.2f}]')
print(f'Scaling Ratio: {ratio:.4f}, Padding: ({pad_w:.1f}, {pad_h:.1f})')

## 3. Person Detection & Bounding Box EDA
We run person detection on the sample frame and visualize detections.

In [4]:
detector = PersonDetector()
results = detector.predict(sample_frame, conf=0.45)
boxes = results.boxes

print(f'Detected {len(boxes)} persons in sample frame.')
for i, box in enumerate(boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = float(box.conf[0])
    print(f'  Person #{i+1}: Conf={conf:.2f}, Box=[{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]')

## 4. Static Camera Motion Gating (Auto-Optimization)
Demonstrates the Static Camera Motion Gate filter to bypass redundant inferences on static backgrounds.

In [5]:
motion_gate = StaticCameraMotionGate()
should_infer, motion_score = motion_gate.should_infer(sample_frame)
print(f'Static Camera Motion Gate -> Should Infer: {should_infer}, Motion Ratio: {motion_score:.4f}')

## 5. YOLO Annotation Format Verification
Confirms labels in data/coco_dataset/labels conform to YOLO specification.

In [6]:
coco_label_dir = Path('data/coco_dataset/labels')
if not coco_label_dir.exists():
    coco_label_dir = Path('../data/coco_dataset/labels')

label_files = list(coco_label_dir.glob('*.txt'))
print(f'Found {len(label_files)} YOLO annotation files.')
if label_files:
    with open(label_files[0], 'r') as f:
        lines = f.readlines()
    print(f'Sample file ({label_files[0].name}):')
    for line in lines:
        print(' ', line.strip())
print('\nAll EDA checks completed successfully!')